# **Fase 2 — Integración espacial de la movilidad en QGIS**

## QUÉ HACE ESTE NOTEBOOK
---------------------
Continúa a partir de las salidas de `Limpieza_Escenarios.ipynb` (Fase 1).
Igual que en la Fase 1, cada decisión queda registrada en una bitácora
(`bitacora`) que se exporta al final como `reporte_auditoria_fase2.txt`,
para poder citarla directamente en la metodología de la tesis.

1. Diagnóstico espacial: cuántos hexágonos son atravesados por alguna
   ruta, cuántos no, y cuántos por más de una (Paso 10a).
2. Verificación de nombres entre `Rutas_Movilidad` y la tabla de
   movilidad (Paso 10b).
3. Agregación del índice de movilidad por hexágono, hora y día (Paso 10c).
4. Conversión a formato temporal largo, con fecha distinta por día
   (Paso 11).
5. Generación de la capa espacial temporal por día, lista para el
   Controlador Temporal de QGIS (Paso 12).

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# CONFIGURACIÓN 
BASE = Path(r"C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial")
CARPETA_SALIDA = BASE / "salidas"
CARPETA_HEX = CARPETA_SALIDA / "hexagonos_movilidad"
CARPETA_HEX.mkdir(parents=True, exist_ok=True)

RUTA_INTERSECCION = BASE / r"RiesgosCapas\Movilidad\fragmementos_interseccion.csv"
RUTA_HEXAGONOS = BASE / r"RiesgosCapas\hex_procesoexp.geojson"
RUTA_BITACORA = CARPETA_HEX / "reporte_auditoria_fase2.txt"

CAMPO_HEX = "id"
CAMPO_AVENIDA_INTERSECCION = "Nombre"
CAMPO_AVENIDA_MOVILIDAD = "avenida"

DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
COLUMNAS_HORAS = [f"{h:02d}h" for h in range(24)]

FECHAS = {
    "Lunes": "2026-01-05", "Martes": "2026-01-06", "Miércoles": "2026-01-07",
    "Jueves": "2026-01-08", "Viernes": "2026-01-09", "Sábado": "2026-01-10",
    "Domingo": "2026-01-11",
}

# Misma función de bitácora que en la Fase 1: centraliza el registro de
# cada decisión para no olvidar documentar algo importante.
bitacora = []

def log(mensaje, mostrar=True):
    bitacora.append(mensaje)
    if mostrar:
        print(mensaje)

interseccion = pd.read_csv(RUTA_INTERSECCION, encoding="utf-8-sig")
hexagonos = gpd.read_file(RUTA_HEXAGONOS)[["id", "geometry"]]
hexagonos["id"] = hexagonos["id"].astype(int)

log("FASE 2 - INTEGRACIÓN ESPACIAL DE LA MOVILIDAD")
log(f"Total de hexágonos en la malla: {hexagonos['id'].nunique()}")
log(f"Total de fragmentos en la intersección Rutas_Movilidad x hexágonos: {len(interseccion)}")

FASE 2 - INTEGRACIÓN ESPACIAL DE LA MOVILIDAD
Total de hexágonos en la malla: 169
Total de fragmentos en la intersección Rutas_Movilidad x hexágonos: 49


## Paso 10a — Diagnóstico espacial

Antes de decidir cómo agregar los valores de movilidad dentro de cada
hexágono, se examina la geometría resultante de la intersección: cuántos
hexágonos tienen cobertura directa, cuántos no, y cuántos son atravesados
por más de una ruta. El criterio de agregación (Paso 10c) se define **a
partir de** este diagnóstico, no antes.

In [2]:
log("PASO 10a - DIAGNÓSTICO ESPACIAL")

total_hexagonos = hexagonos["id"].nunique()
hex_con_ruta = interseccion[CAMPO_HEX].nunique()
hex_sin_ruta = total_hexagonos - hex_con_ruta

rutas_por_hex = interseccion.groupby(CAMPO_HEX)[CAMPO_AVENIDA_INTERSECCION].nunique()
hex_con_cruce = (rutas_por_hex > 1).sum()

log(f"Hexágonos tocados por al menos una ruta: {hex_con_ruta} de {total_hexagonos} "
    f"({100*hex_con_ruta/total_hexagonos:.1f}%)")
log(f"Hexágonos SIN ninguna ruta: {hex_sin_ruta} de {total_hexagonos} "
    f"({100*hex_sin_ruta/total_hexagonos:.1f}%)")
log(f"Hexágonos con más de una ruta (cruces): {hex_con_cruce} "
    f"({100*hex_con_cruce/total_hexagonos:.1f}%)")

log("\nDecisión (criterio de agregación, Paso 10c): dado que los cruces "
    "son un caso minoritario (menos del 3% de los hexágonos), se usa un "
    "promedio aritmético simple entre las rutas presentes en vez de una "
    "ponderación por longitud de fragmento. El efecto práctico de esta "
    "simplificación sobre el resultado final es marginal, y se documenta "
    "aquí como decisión deliberada, no como una omisión.")

log("\nDecisión (hexágonos sin ninguna ruta): se conservan con movilidad "
    "NULL. No se interpola ni se asigna un valor por cercanía (vecino más "
    "próximo, buffer de influencia), porque no existe evidencia observada "
    "de movilidad en esas celdas y cualquier método de relleno introduce "
    "un supuesto no verificado. Estos hexágonos se caracterizarán más "
    "adelante mediante una variable complementaria de accesibilidad al "
    "transporte público (metro, Metrobús, RTP).")

PASO 10a - DIAGNÓSTICO ESPACIAL
Hexágonos tocados por al menos una ruta: 39 de 169 (23.1%)
Hexágonos SIN ninguna ruta: 130 de 169 (76.9%)
Hexágonos con más de una ruta (cruces): 8 (4.7%)

Decisión (criterio de agregación, Paso 10c): dado que los cruces son un caso minoritario (menos del 3% de los hexágonos), se usa un promedio aritmético simple entre las rutas presentes en vez de una ponderación por longitud de fragmento. El efecto práctico de esta simplificación sobre el resultado final es marginal, y se documenta aquí como decisión deliberada, no como una omisión.

Decisión (hexágonos sin ninguna ruta): se conservan con movilidad NULL. No se interpola ni se asigna un valor por cercanía (vecino más próximo, buffer de influencia), porque no existe evidencia observada de movilidad en esas celdas y cualquier método de relleno introduce un supuesto no verificado. Estos hexágonos se caracterizarán más adelante mediante una variable complementaria de accesibilidad al transporte público (met

## Paso 10b — Verificación de nombres

Se comprueba que todos los nombres de `Rutas_Movilidad` (campo `Nombre`)
tengan coincidencia exacta en la tabla de movilidad (campo `avenida`). Un
desajuste no genera error — `pandas` deja esa fila en `NaN` y `.mean()` la
ignora sin avisar, lo que produciría un promedio de hexágono calculado con
menos rutas de las que realmente lo atraviesan. Queda como salvaguarda
para el futuro (por ejemplo, si se agrega una avenida nueva).

In [3]:
log("PASO 10b - VALIDACIÓN DE NOMBRES (join Nombre <-> avenida)")

movilidad_referencia = pd.read_csv(CARPETA_SALIDA / f"movilidad_{DIAS[0]}.csv", encoding="utf-8-sig")
nombres_interseccion = set(interseccion[CAMPO_AVENIDA_INTERSECCION].unique())
nombres_movilidad = set(movilidad_referencia[CAMPO_AVENIDA_MOVILIDAD].unique())
sin_match = nombres_interseccion - nombres_movilidad

if sin_match:
    log(f"⚠ Sin coincidencia exacta (quedarían como NaN, ignorados por .mean() sin aviso): {sin_match}")
else:
    log("Todos los nombres de 'Rutas_Movilidad' tienen coincidencia exacta en la tabla de movilidad.")

assert not sin_match, "Hay nombres sin coincidencia exacta: revisa acentos/espacios antes de continuar."


PASO 10b - VALIDACIÓN DE NOMBRES (join Nombre <-> avenida)
Todos los nombres de 'Rutas_Movilidad' tienen coincidencia exacta en la tabla de movilidad.


## Paso 10c — Agregación de movilidad por hexágono

Se aplica el criterio definido en el Paso 10a (promedio aritmético simple
para hexágonos con más de una ruta), para los 7 días de la semana.

In [4]:
log("PASO 10c - AGREGACIÓN POR HEXÁGONO (7 días)")

for dia in DIAS:
    movilidad = pd.read_csv(CARPETA_SALIDA / f"movilidad_{dia}.csv", encoding="utf-8-sig")

    tabla = interseccion.merge(
        movilidad, left_on=CAMPO_AVENIDA_INTERSECCION, right_on=CAMPO_AVENIDA_MOVILIDAD, how="left"
    )
    resultado = tabla.groupby(CAMPO_HEX)[COLUMNAS_HORAS].mean().reset_index()

    ruta = CARPETA_HEX / f"hex_movilidad_{dia}.csv"
    resultado.to_csv(ruta, index=False, encoding="utf-8-sig")
    log(f"  {dia}: {ruta.name} ({len(resultado)} hexágonos)")

log(f"\nArchivos guardados en: {CARPETA_HEX}")

PASO 10c - AGREGACIÓN POR HEXÁGONO (7 días)
  Lunes: hex_movilidad_Lunes.csv (39 hexágonos)
  Martes: hex_movilidad_Martes.csv (39 hexágonos)
  Miércoles: hex_movilidad_Miércoles.csv (39 hexágonos)
  Jueves: hex_movilidad_Jueves.csv (39 hexágonos)
  Viernes: hex_movilidad_Viernes.csv (39 hexágonos)
  Sábado: hex_movilidad_Sábado.csv (39 hexágonos)
  Domingo: hex_movilidad_Domingo.csv (39 hexágonos)

Archivos guardados en: C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\salidas\hexagonos_movilidad


## Paso 11 — Conversión a formato temporal (ancho → largo)

Cada `hex_movilidad_{día}.csv` se convierte a formato largo: una fila por
combinación hexágono-hora, con `fecha_hora` construida a partir de una
fecha ficticia distinta por día (5 al 11 de enero de 2026) — esto permite
diferenciar los 7 días si en algún momento se cargan varias capas juntas
en el controlador temporal de QGIS.

In [5]:
log("PASO 11 - CONVERSIÓN A FORMATO TEMPORAL (7 días)")

for dia in DIAS:
    archivo = CARPETA_HEX / f"hex_movilidad_{dia}.csv"
    salida = CARPETA_HEX / f"movilidad_temporal_{dia}.csv"

    df = pd.read_csv(archivo)
    df_largo = df.melt(
        id_vars="id", value_vars=COLUMNAS_HORAS, var_name="hora", value_name="movilidad"
    )
    df_largo["fecha_hora"] = pd.to_datetime(
        FECHAS[dia] + " " + df_largo["hora"].str.replace("h", ":00")
    )
    df_largo = df_largo.sort_values(["fecha_hora", "id"]).reset_index(drop=True)
    df_largo.to_csv(salida, index=False, encoding="utf-8-sig")

    log(f"  {dia}: {salida.name} ({len(df_largo)} registros, fecha base {FECHAS[dia]})")

PASO 11 - CONVERSIÓN A FORMATO TEMPORAL (7 días)
  Lunes: movilidad_temporal_Lunes.csv (936 registros, fecha base 2026-01-05)
  Martes: movilidad_temporal_Martes.csv (936 registros, fecha base 2026-01-06)
  Miércoles: movilidad_temporal_Miércoles.csv (936 registros, fecha base 2026-01-07)
  Jueves: movilidad_temporal_Jueves.csv (936 registros, fecha base 2026-01-08)
  Viernes: movilidad_temporal_Viernes.csv (936 registros, fecha base 2026-01-09)
  Sábado: movilidad_temporal_Sábado.csv (936 registros, fecha base 2026-01-10)
  Domingo: movilidad_temporal_Domingo.csv (936 registros, fecha base 2026-01-11)


## Paso 12 — Generación de la capa espacial temporal

Cada hexágono se replica 24 veces (una por hora), uniendo el valor de
movilidad por `id` + `fecha_hora`. Los hexágonos sin cobertura de ruta
quedan con `movilidad = NULL` (visibles en gris en la animación de QGIS).
Resultado: una capa GeoJSON por día, compatible con el Controlador
Temporal — evita generar 24 (o 168) capas estáticas independientes.

In [6]:
log("PASO 12 - CAPAS ESPACIALES TEMPORALES (7 días)")

for dia in DIAS:
    ruta_movilidad = CARPETA_HEX / f"movilidad_temporal_{dia}.csv"
    ruta_salida = CARPETA_HEX / f"HEX_TEMPORAL_{dia}.geojson"

    mov = pd.read_csv(ruta_movilidad)
    mov["id"] = mov["id"].astype(int)
    mov["fecha_hora"] = pd.to_datetime(mov["fecha_hora"])

    horas_unicas = sorted(mov["fecha_hora"].unique())
    piezas = []
    for fecha in horas_unicas:
        temp = hexagonos.copy()
        temp["fecha_hora"] = fecha
        temp["hora"] = pd.to_datetime(fecha).strftime("%Hh")
        piezas.append(temp)
    base_temporal = pd.concat(piezas, ignore_index=True)

    resultado = base_temporal.merge(
        mov[["id", "fecha_hora", "movilidad"]], on=["id", "fecha_hora"], how="left"
    )
    resultado = gpd.GeoDataFrame(resultado, geometry="geometry", crs=hexagonos.crs)
    resultado.to_file(ruta_salida, driver="GeoJSON")

    n_con_dato = resultado["movilidad"].notna().sum()
    log(f"  {dia}: {ruta_salida.name} ({len(resultado)} registros, "
        f"{n_con_dato} con valor de movilidad, {len(resultado)-n_con_dato} nulos)")

PASO 12 - CAPAS ESPACIALES TEMPORALES (7 días)
  Lunes: HEX_TEMPORAL_Lunes.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Martes: HEX_TEMPORAL_Martes.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Miércoles: HEX_TEMPORAL_Miércoles.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Jueves: HEX_TEMPORAL_Jueves.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Viernes: HEX_TEMPORAL_Viernes.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Sábado: HEX_TEMPORAL_Sábado.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)
  Domingo: HEX_TEMPORAL_Domingo.geojson (4056 registros, 936 con valor de movilidad, 3120 nulos)


## Paso 13 — Exportación de la bitácora

Igual que en la Fase 1, todas las decisiones registradas durante este
notebook se guardan como texto plano para citarlas en la metodología.

In [7]:
with open(RUTA_BITACORA, "w", encoding="utf-8") as f:
    f.write("\n".join(bitacora))

print(f"\nBitácora guardada en: {RUTA_BITACORA}")
print("\nFIN DEL NOTEBOOK - Fase 2 completada.")


Bitácora guardada en: C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\salidas\hexagonos_movilidad\reporte_auditoria_fase2.txt

FIN DEL NOTEBOOK - Fase 2 completada.
